In [22]:
import csv
import re
import numpy as np
import pickle

### Read the CSV File

In [23]:
articles = []
try:
    with open('articles.csv', mode='r') as file:
        reader = csv.DictReader(file)
        for row in reader:
            articles.append(row)
except FileNotFoundError:
    print("Error: 'articles.csv' not found.")
    exit()


### Clean Article Content

In [24]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    return words

all_cleaned_contents = []
for art in articles:
    cleaned = clean_text(art['content'])
    all_cleaned_contents.append(cleaned)

### Bag of Words

In [25]:
vocabulary = set()
for word_list in all_cleaned_contents:
    for word in word_list:
        vocabulary.add(word)

vocabulary = sorted(list(vocabulary))

### Vectorization

In [26]:
article_vectors = []
for word_list in all_cleaned_contents:
    vector = [1 if word in word_list else 0 for word in vocabulary]
    article_vectors.append(vector)

vectors_np = np.array(article_vectors, dtype=float)

### Calculate Cosine Similarity

In [27]:
num_articles = len(articles)
similarity_matrix = np.zeros((num_articles, num_articles))

for i in range(num_articles):
    for j in range(num_articles):
        vec_a = vectors_np[i]
        vec_b = vectors_np[j]
        
        dot_product = np.dot(vec_a, vec_b)
        norm_a = np.linalg.norm(vec_a)
        norm_b = np.linalg.norm(vec_b)
        
        if norm_a == 0 or norm_b == 0:
            similarity_matrix[i][j] = 0.0
        else:
            # Cosine Similarity Formula
            similarity_matrix[i][j] = dot_product / (norm_a * norm_b)

### Save to PKL

In [28]:
with open('similarities.pkl', 'wb') as f:
    pickle.dump(similarity_matrix, f)

### Find Similar Articles

In [29]:
def get_top_3_similar(target_id):
    target_idx = -1
    for idx, art in enumerate(articles):
        if art['id'] == str(target_id):
            target_idx = idx
            break
            
    if target_idx == -1:
        return None

    scores = []
    for idx, score in enumerate(similarity_matrix[target_idx]):
        if idx != target_idx:
            scores.append((idx, score))
    
    scores.sort(key=lambda x: x[1], reverse=True)
    
    results = []
    for i in range(min(3, len(scores))):
        idx, score = scores[i]
        results.append((articles[idx]['title'], score))
    return results


## Main  

In [ ]:
def main():
    print("System Ready. Similarity matrix saved.")
    
    keep_running = True
    while keep_running:
        # Show available IDs
        ids = [art['id'] for art in articles]
        print(f"\nAvailable IDs: {', '.join(ids)}")
        
        user_choice = input("Enter an Article ID to analyze: ").strip()
        
        matches = get_top_3_similar(user_choice)
        
        if matches is None:
            print(f"Error: ID '{user_choice}' not found.")
        else:
            print(f"\nTop 3 matches for ID {user_choice}:")
            for i, (title, score) in enumerate(matches, 1):
                print(f"{i}. {title} (Similarity: {score*100:.1f}%)")
        
        print("\n------------------------------------")
        choice = input("Would you like to enter another ID? (yes/no): ").strip().lower()
        
        if choice != 'yes' and choice != 'y':
            print("Exiting program. Goodbye!")
            keep_running = False

if __name__ == "__main__":
    main()

System Ready. Similarity matrix saved.

Available IDs: 1, 2, 3, 4, 5, 6, 7, 8

Top 3 matches for ID 4:
1. Quantum Computing Breakthrough (Similarity: 14.7%)
2. AI in Car Manufacturing (Similarity: 13.6%)
3. Titan Moon Origins (Similarity: 13.1%)

------------------------------------
Exiting program. Goodbye!
